# Backtest Analysis — Elliott Wave Trading Agent

Run backtests across multiple assets and timeframes,
then analyze results to find where the pattern works best.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from data.yfinance_provider import YFinanceProvider
from signals.wave3_rsi import Wave3RSISignal
from backtest.engine import BacktestEngine
from backtest.metrics import calculate_metrics
from backtest.report import generate_report, print_report

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Configure & Run Backtest

In [ ]:
# Assets and timeframes to test
ASSETS = ['SPY', 'QQQ', 'AAPL', 'MSFT', 'NVDA', 'GLD']
TIMEFRAMES = ['1d']
START = '2020-01-01'
END = '2024-12-31'
INITIAL_CAPITAL = 10_000

provider = YFinanceProvider()
signal = Wave3RSISignal()
engine = BacktestEngine(
    signal_generator=signal,
    initial_capital=INITIAL_CAPITAL,
    risk_per_trade=0.02,
)

In [ ]:
results = {}
for symbol in ASSETS:
    for tf in TIMEFRAMES:
        print(f'Backtesting {symbol} ({tf})...')
        try:
            df = provider.fetch_ohlcv(symbol, timeframe=tf, start=START, end=END)
            if len(df) < 50:
                print(f'  Skipping: insufficient data ({len(df)} bars)')
                continue
            result = engine.run(df, symbol)
            results[(symbol, tf)] = result
            print(f'  {result.num_trades} trades, {result.total_return_pct:+.1f}% return')
        except Exception as e:
            print(f'  Error: {e}')

## 2. Ranked Results

In [ ]:
reports = generate_report(results)
report_str = print_report(reports)
print(report_str)

## 3. Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for (symbol, tf), result in results.items():
    equity = result.equity_curve
    ax.plot(range(len(equity)), equity, label=f'{symbol} ({tf})', linewidth=1.5)

ax.axhline(y=INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Equity Curves — Wave 3 + RSI Strategy')
ax.set_xlabel('Bars')
ax.set_ylabel('Equity ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Metrics Comparison

In [ ]:
# Build comparison table
rows = []
for r in reports:
    m = r.metrics
    rows.append({
        'Symbol': r.symbol,
        'Timeframe': r.timeframe,
        'Trades': m.total_trades,
        'Win Rate': f'{m.win_rate:.1%}',
        'Profit Factor': f'{m.profit_factor:.2f}',
        'Sharpe': f'{m.sharpe_ratio:.2f}',
        'Max DD': f'{m.max_drawdown_pct:.1%}',
        'Return': f'{m.total_return_pct:+.1f}%',
        'Score': f'{m.composite_score():.3f}',
    })

comparison = pd.DataFrame(rows)
comparison

## 5. AI Analysis (Optional)

Uncomment below to get AI-powered analysis of the results.

In [ ]:
# from agents.research_agent import ResearchAgent
# researcher = ResearchAgent()
# analysis = researcher.analyze_backtest_results(reports)
# print(analysis.summary)
# print('\nBest assets:', analysis.best_assets)